# Entertainment Recommendation & Curation System
## DSA 2020A – Lab 2 | Multi-Agent AI Demo (LangGraph + Groq)

### Architecture
```
User Input
    │
    ▼
┌──────────────┐
│  SUPERVISOR  │  ← LangGraph StateGraph node
│  (router)    │    Routes tasks, decides FINISH
└──────┬───────┘
       │ conditional edges
  ┌────┴────┬──────────┬───────────┬──────────┬────────────┐
  ▼         ▼          ▼           ▼          ▼            ▼
Profiler Researcher  Matcher   Curator   Planner    Reviewer
         (search)   (score)  (HITL ✋) (schedule) (critique)
  │         │          │           │          │            │
  └─────────┴──────────┴─────────→ Supervisor (loop back)
```

### Required Technical Elements Covered
| Requirement | Implementation |
|-------------|----------------|
| Supervisor/Orchestrator | `supervisor_node` with structured output routing |
| 3-5 Specialized Agents | Profiler, Researcher, Matcher, Curator, Planner, Reviewer |
| Shared State/Memory | `AgentState` TypedDict + `MemorySaver` checkpointer |
| Tool Integration | DuckDuckGo Search, genre_matcher, schedule_planner |
| Human-in-the-Loop | `interrupt_before=["curator"]` + user input |
| Reflection/Critique | Quality Reviewer node (independent critic) |
| Streaming | `graph.stream()` with live agent output |
| Termination | Supervisor returns `FINISH` → graph ends |

In [ ]:
# Install dependencies (run once)
# !pip install langgraph langchain-groq langchain-community duckduckgo-search python-dotenv

In [ ]:
import os
import json
import operator
from typing import TypedDict, Annotated, Literal
from dotenv import load_dotenv

load_dotenv()

# Paste your Groq API key here if not using .env file
# os.environ['GROQ_API_KEY'] = 'gsk_your_key_here'

print('GROQ_API_KEY set:', bool(os.environ.get('GROQ_API_KEY')))

## Step 1: LLM + Tools

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import create_react_agent
from langgraph.types import interrupt
from pydantic import BaseModel

llm = ChatGroq(
    model='llama-3.3-70b-versatile',
    api_key=os.environ.get('GROQ_API_KEY'),
    temperature=0.7,
    max_tokens=2048,
)

@tool
def genre_matcher(preferences: str) -> str:
    """Maps user preference keywords to entertainment genres. Input: preference description."""
    mapping = {
        'action':      ['Action', 'Thriller', 'Adventure'],
        'comedy':      ['Comedy', 'Sitcom', 'Stand-up'],
        'romance':     ['Romance', 'Drama', 'Rom-com'],
        'sci-fi':      ['Science Fiction', 'Cyberpunk', 'Space Opera'],
        'horror':      ['Horror', 'Psychological Thriller'],
        'documentary': ['Documentary', 'True Crime', 'Nature'],
        'fantasy':     ['Fantasy', 'Epic Fantasy', 'Mythology'],
        'animation':   ['Animation', 'Anime'],
        'music':       ['Pop', 'Jazz', 'Hip-hop', 'Rock'],
        'gaming':      ['RPG', 'Indie', 'Action-Adventure'],
        'relaxing':    ['Slice of Life', 'Nature', 'Ambient'],
        'exciting':    ['Action', 'Thriller', 'Competition'],
    }
    result = {k: v for k, v in mapping.items() if k in preferences.lower()}
    return json.dumps(result or {'general': ['Drama', 'Comedy', 'Trending']}, indent=2)

@tool
def schedule_planner(hours_available: str) -> str:
    """Creates a time-slot entertainment schedule. Input: hours as a string e.g. '3'."""
    try:
        hours = float(hours_available.strip().split()[0])
    except Exception:
        hours = 3.0
    slots, remaining = [], hours
    if remaining >= 2.0:
        slots.append({'slot': 'Main Feature', 'type': 'Movie or 2-episode binge', 'duration': '~2 hours'})
        remaining -= 2.0
    if remaining >= 0.75:
        slots.append({'slot': 'Mid-session', 'type': 'Single episode', 'duration': '~45 min'})
    if remaining >= 0.5:
        slots.append({'slot': 'Wind-down', 'type': 'Music/Podcast', 'duration': '~30 min'})
    return json.dumps({'available_hours': hours, 'schedule': slots}, indent=2)

search_tool = DuckDuckGoSearchRun()
print('LLM + 3 tools ready')

## Step 2: Shared State & Supervisor

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]
    user_request: str
    next: str

MEMBERS   = ['profiler', 'researcher', 'matcher', 'curator', 'planner', 'reviewer']

class Route(BaseModel):
    next: Literal['profiler', 'researcher', 'matcher', 'curator', 'planner', 'reviewer', 'FINISH']

SUPERVISOR_PROMPT = """You are the supervisor of an entertainment recommendation team.
Workers: profiler → researcher → matcher → curator → planner → reviewer → FINISH.
Route strictly in this order. Return FINISH only after reviewer has responded."""

def supervisor_node(state: AgentState) -> dict:
    msgs   = [SystemMessage(content=SUPERVISOR_PROMPT)] + state['messages']
    result = llm.with_structured_output(Route).invoke(msgs)
    return {'next': result.next}

print('Supervisor configured')

## Step 3: Specialized Worker Agents

In [ ]:
def make_node(agent, name: str):
    def node(state: AgentState) -> dict:
        result = agent.invoke({'messages': state['messages']})
        return {'messages': [AIMessage(content=result['messages'][-1].content, name=name)]}
    return node

profiler_agent = create_react_agent(llm, tools=[genre_matcher], prompt=(
    'You are an entertainment preference profiler. Build a detailed JSON taste profile '
    '(genres, formats, mood, estimated_hours) from the user request. Use the genre_matcher tool.'
))

researcher_agent = create_react_agent(llm, tools=[search_tool], prompt=(
    'You are a content discovery expert. Search for 6-8 highly-rated entertainment options '
    'matching the preference profile. Return: title, type, rating, platform, 1-sentence description.'
))

matcher_agent = create_react_agent(llm, tools=[genre_matcher], prompt=(
    'You are a recommendation engine. Score each researched item 1-10 against the taste profile. '
    'Provide one-sentence reasoning per item. Return top 5 ranked by score.'
))

curator_agent = create_react_agent(llm, tools=[search_tool], prompt=(
    'You are a diversity curator. Ensure 2+ formats, max 2 per genre. '
    'Add one [SERENDIPITY PICK] outside the user usual taste. Return final 5-item list.'
))

planner_agent = create_react_agent(llm, tools=[schedule_planner], prompt=(
    'You are an entertainment concierge. Call schedule_planner with user available hours, '
    'then build a plan: Title | Type | Platform | Duration | Time Slot | Why it is for you. '
    'Mark serendipity as *** SURPRISE PICK ***.'
))

reviewer_agent = create_react_agent(llm, tools=[], prompt=(
    'You are a quality reviewer. Verify titles/platforms are real, check preference alignment. '
    'Correct errors. Append a 2-sentence WHY THIS PLAN WORKS FOR YOU summary.'
))

print('6 specialized agents created')

## Step 4: Build the LangGraph

In [ ]:
def curator_with_hitl(state: AgentState) -> dict:
    # Run curator
    result  = curator_agent.invoke({'messages': state['messages']})
    curated = result['messages'][-1].content
    # Pause for human approval
    feedback = interrupt({'curated_list': curated, 'prompt': 'Approve or suggest changes:'})
    content  = curated + (f'\n[Human: {feedback}]' if feedback and str(feedback).strip() else '')
    return {'messages': [AIMessage(content=content, name='curator')]}

def route_next(state: AgentState) -> str:
    return state['next']

graph_builder = StateGraph(AgentState)
graph_builder.add_node('supervisor',  supervisor_node)
graph_builder.add_node('profiler',    make_node(profiler_agent,   'profiler'))
graph_builder.add_node('researcher',  make_node(researcher_agent, 'researcher'))
graph_builder.add_node('matcher',     make_node(matcher_agent,    'matcher'))
graph_builder.add_node('curator',     curator_with_hitl)
graph_builder.add_node('planner',     make_node(planner_agent,    'planner'))
graph_builder.add_node('reviewer',    make_node(reviewer_agent,   'reviewer'))

graph_builder.add_edge(START, 'supervisor')
graph_builder.add_conditional_edges(
    'supervisor', route_next,
    {'profiler': 'profiler', 'researcher': 'researcher', 'matcher': 'matcher',
     'curator': 'curator', 'planner': 'planner', 'reviewer': 'reviewer', 'FINISH': END}
)
for member in MEMBERS:
    graph_builder.add_edge(member, 'supervisor')

checkpointer = MemorySaver()
graph = graph_builder.compile(
    checkpointer=checkpointer,
    interrupt_before=['curator'],  # HUMAN-IN-THE-LOOP pause
)
print('LangGraph compiled successfully')

## Step 5: Run Demo Scenario — Sci-Fi Movie Night

In [ ]:
USER_REQUEST = "I love sci-fi and action movies, have about 3 hours tonight."
config       = {'configurable': {'thread_id': 'demo-1'}}
state        = {'messages': [HumanMessage(content=USER_REQUEST)], 'user_request': USER_REQUEST, 'next': ''}

print(f'User: {USER_REQUEST}\n')
print('Streaming agent actions...\n' + '='*60)

seen = set()

# Phase 1: run until curator interrupt
for event in graph.stream(state, config, stream_mode='values'):
    msgs = event.get('messages', [])
    if msgs:
        m = msgs[-1]
        label = getattr(m, 'name', None) or type(m).__name__
        if label not in MEMBERS:
            continue
        key = id(m)
        if key in seen:
            continue
        seen.add(key)
        if hasattr(m, 'content') and m.content:
            print(f'[{label.upper()}]:\n{m.content[:400]}\n')

print('\n>>> HUMAN-IN-THE-LOOP PAUSE — curator awaiting your approval <<<')

In [ ]:
# Approve the curated list (leave empty to approve, or type feedback)
human_feedback = ''  # Change this to provide feedback, e.g. 'Swap the horror pick for a comedy'

# Resume after human input
resume_input = {'messages': [HumanMessage(content=human_feedback or 'Approved, looks great!')]}

for event in graph.stream(resume_input, config, stream_mode='values'):
    msgs = event.get('messages', [])
    if msgs:
        m = msgs[-1]
        label = getattr(m, 'name', type(m).__name__)
        if hasattr(m, 'content') and m.content:
            print(f'[{label.upper()}]: {m.content[:400]}\n')

In [ ]:
# Print final plan
final_state = graph.get_state(config)
final_msgs  = final_state.values.get('messages', [])
if final_msgs:
    print('\n' + '='*60)
    print('FINAL ENTERTAINMENT PLAN')
    print('='*60)
    print(final_msgs[-1].content)

## Summary

This demo showed all required multi-agent features:
- **Supervisor routing** — structured output decides which agent acts next
- **Specialized workers** — 6 agents with distinct roles and tools
- **Shared state** — `AgentState` with message history across all nodes
- **3 tools** — DuckDuckGo Search, genre_matcher, schedule_planner
- **Human-in-the-loop** — graph paused at `interrupt_before=['curator']`
- **Reflection loop** — Quality Reviewer independently critiques the plan
- **Streaming** — `graph.stream()` shows live agent output
- **Termination** — supervisor returns `FINISH` when reviewer completes